Field(...) 适合声明通用约束

但有些业务规则无法仅靠 Field 表达，例如：
- 邮箱统一转为小写；
- 禁止使用保留用户名；
- 开始时间必须早于结束时间；
- 两次输入的密码必须一致；
- 折扣价不能高于原价。
  
Pydantic v2 提供:`field_validator` 和 `model_validator`来实现。

In [ ]:
# field_validator
from pydantic import BaseModel, field_validator

class UserDTO(BaseModel):
    email: str
    # 字段验证器默认相当于 mode="after",表示先进行类型解析，再执行验证器
    @field_validator("email")
    @classmethod
    def normalize_email(cls, value: str) -> str:
        normalized = value.strip().lower() # 邮箱统一转为小写

        # 抛出 ValueError
        if "@" not in normalized:
            raise ValueError("Invalid email")
        # # 必须返回最终字段值
        return normalized

In [ ]:
# mode="before"
# 在 Pydantic 类型解析前执行，所以拿到的是原始输入；适合自定义字符串转换、兼容旧格式、在解析前清理特殊值
@field_validator("age", mode="before")
@classmethod
def parse_age(cls, value):
    ...


#### 同时验证多个字段

In [ ]:
@field_validator("name", "nickname")
@classmethod
def strip_text(cls, value: str) -> str:
    return value.strip()

### model_validator
跨字段规则使用模型验证器

In [ ]:
from typing import Self

from pydantic import BaseModel, model_validator


class RegisterDTO(BaseModel):
    password: str
    password_confirm: str

    @model_validator(mode="after")
    def validate_passwords(self) -> Self:
        if self.password != self.password_confirm:
            raise ValueError("Passwords do not match")

        return self

In [ ]:
{
  "password": "password123",
  "password_confirm": "password456"
}

每个字段单独看都是合法字符串，但二者关系不符合业务规则，所以需要模型级验证。

同时必须返回self

### 大致验证顺序
原始输入
- → mode="before" 验证器
- → 类型解析
- → Field 约束
- → 默认的字段 after 验证器
- → 创建模型
- → model_validator(mode="after")

### 不应放进验证器的逻辑
- 查询数据库；
- 调用远程 API；
- 发送邮件；
- 写入文件；
- 创建用户；
- 修改外部共享状态。

因为验证器可能被多次调用，I/O 会让验证变慢，同步/异步边界混乱，数据校验和业务流程职责混合。